# Day 09 · 偏好聚合：Bradley-Terry、Elo 与排名区间

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 写出 Bradley-Terry 模型，用它从成对偏好得到系统排名
- 解释 Elo 与 Bradley-Terry 的区别（顺序依赖）
- 用 bootstrap 给排名加区间，判断两个系统能否区分
- 量化少量对抗性投票对排名的影响

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 六个心理支持系统的咨询师投票

In [2]:
from course.bt import fit_bradley_terry, bootstrap_bt_ranks, elo_ratings, simulate_pairs
names = ["S1", "S2", "S3", "S4", "S5", "S6"]
strength = np.array([0.8, 0.6, 0.55, 0.2, 0.0, -0.4])
raw = simulate_pairs(strength, 600, seed=9)
votes = [(w, l) for w, l, _ in raw]
s_hat = fit_bradley_terry(votes, 6)
pd.DataFrame({"system": names, "true": strength - strength.mean(), "estimated": s_hat}).round(3)

,system,true,estimated
0,S1,0.508,0.752
1,S2,0.308,0.231
2,S3,0.258,0.319
3,S4,-0.092,0.049
4,S5,-0.292,-0.485
5,S6,-0.692,-0.867


**练习 9.1**：用估计的强度计算 S1 胜过 S3 的概率。

In [3]:
ans_9_1 = None
check("练习 9.1 P(S1 胜 S3)", ans_9_1, 1 / (1 + np.exp(-(s_hat[0] - s_hat[2]))), tol=1e-6)

[ 待完成 ] 练习 9.1 P(S1 胜 S3)


## 2. 排名区间

In [4]:
ranks, _ = bootstrap_bt_ranks(votes, 6, n_boot=400, seed=1)
ri = pd.DataFrame({"system": names, "median": np.median(ranks, 0), "lo": np.quantile(ranks, .025, 0), "hi": np.quantile(ranks, .975, 0)})
ri

,system,median,lo,hi
0,S1,1.0,1.0,1.0
1,S2,3.0,2.0,4.0
2,S3,2.0,2.0,4.0
3,S4,4.0,2.0,4.0
4,S5,5.0,5.0,6.0
5,S6,6.0,5.0,6.0


**练习 9.2**:S2 和 S3 的排名区间是否重叠？填 `True` 或 `False`。

In [5]:
ans_9_2 = None
check("练习 9.2", ans_9_2, bool(ri.loc[1, "hi"] >= ri.loc[2, "lo"] and ri.loc[2, "hi"] >= ri.loc[1, "lo"]))

[ 待完成 ] 练习 9.2


## 3. Elo 依赖比赛顺序

把同一批投票打乱顺序，重新计算 Elo，看 S2 和 S3 的分数差如何变化。

In [6]:
diffs = []
for seed in range(30):
    order = np.random.default_rng(seed).permutation(len(votes))
    r = elo_ratings([votes[i] for i in order], 6)
    diffs.append(r[1] - r[2])
print(f"S2 - S3 的 Elo 差: 最小 {min(diffs):.1f}, 最大 {max(diffs):.1f}")
print("Bradley-Terry 估计不随顺序变化:", round(s_hat[1] - s_hat[2], 3))

S2 - S3 的 Elo 差: 最小 -89.6, 最大 111.6
Bradley-Terry 估计不随顺序变化: -0.087


**练习 9.3**：在 30 种顺序中，有几种顺序下 Elo 认为 S3 比 S2 强（差值小于 0)?

In [7]:
ans_9_3 = None
check("练习 9.3", ans_9_3, int(sum(d < 0 for d in diffs)))

[ 待完成 ] 练习 9.3


## 4. 对抗性投票

加入 10% 的额外投票，全部让最弱的 S6 胜过 S4。看排名变化。

In [8]:
n_bad = int(0.10 * len(votes))
bad = [(5, 3)] * n_bad
s_bad = fit_bradley_terry(votes + bad, 6)
print("原排名:", [names[i] for i in np.argsort(-s_hat)])
print("加入对抗投票后:", [names[i] for i in np.argsort(-s_bad)])

原排名: ['S1', 'S3', 'S2', 'S4', 'S5', 'S6']
加入对抗投票后: ['S1', 'S3', 'S2', 'S6', 'S4', 'S5']


**练习 9.4**：加入对抗投票后，S6 的名次上升了几位？

In [9]:
_rank = lambda s, i: int(np.argsort(np.argsort(-s))[i]) + 1
ans_9_4 = None
check("练习 9.4", ans_9_4, _rank(s_hat, 5) - _rank(s_bad, 5))

[ 待完成 ] 练习 9.4


## 口述题（录音，每题 90 秒）

1. 为什么排行榜要给排名区间？
   <details><summary>参考要点</summary>名次是对强度估计的离散化，强度差小于估计误差时名次不稳定。bootstrap 排名区间直接显示哪些名次差异有数据支持。</details>
2. Elo 和 Bradley-Terry 有什么区别？
   <details><summary>参考要点</summary>Elo 逐场在线更新，结果依赖比赛顺序；Bradley-Terry 用全部数据一次估计，不依赖顺序，适合固定一批模型的评测。</details>

## 研究延伸

在心理支持场景里，咨询师的成对偏好和来访者（或模拟来访者）的成对偏好可能不同。MentalHealthBench 的用户研究也发现专家和用户重视的东西不同。比较这两类偏好得到的排名，本身就是一个效度问题。

In [10]:
summary()

本 notebook 自动批改:0 / 4 通过
